# 🎯 00 · 推荐系统总览与评估指标

> 推荐系统（搜广推）是算法岗最大的方向之一。本节先搭全局框架（召回→粗排→精排→重排→AB），
> 再**手写 AUC / GAUC / NDCG@K / HR@K / Recall@K** 并与 sklearn 对照——评估指标是推荐岗面试第一关。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 推荐系统完整链路是什么？ | 召回→粗排→精排→重排→AB |
| 2 | 排序效果怎么量化？ | AUC（手写 vs sklearn） |
| 3 | 用户级排序怎么评估？ | GAUC |
| 4 | 推荐列表质量怎么评估？ | HR@K / Recall@K / NDCG@K |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 推荐链路总览（必背框架）

In [2]:
stages = [
    ('召回', '从百万级物品粗筛几百~上千候选：协同过滤/双塔/向量检索，快而粗'),
    ('粗排', '轻量模型把候选压到百级：双塔/简单 LR，控制时延'),
    ('精排', '复杂模型逐点精估 CTR/CVR：DeepFM/MMoE，最重的一层'),
    ('重排', '多样性/新鲜度/打散/业务约束，重排最终列表'),
    ('AB 评估', '小流量实验验证线上收益，评估指标驱动迭代'),
]
print('推荐系统五段链路:')
for i, (s, d) in enumerate(stages, 1):
    print(f'  {i}. {s} —— {d}')
print('\n➡️ 面试串讲：从「候选池规模」到「时延预算」讲每层职责与模型选择')

推荐系统五段链路:
  1. 召回 —— 从百万级物品粗筛几百~上千候选：协同过滤/双塔/向量检索，快而粗
  2. 粗排 —— 轻量模型把候选压到百级：双塔/简单 LR，控制时延
  3. 精排 —— 复杂模型逐点精估 CTR/CVR：DeepFM/MMoE，最重的一层
  4. 重排 —— 多样性/新鲜度/打散/业务约束，重排最终列表
  5. AB 评估 —— 小流量实验验证线上收益，评估指标驱动迭代

➡️ 面试串讲：从「候选池规模」到「时延预算」讲每层职责与模型选择


## 2️⃣ 构造点击/打分数据

In [3]:
# 模拟 100 用户 × 50 物品的评分/点击
n_users, n_items = 100, 50
true_scores = rng.normal(0, 1, (n_users, n_items))
# 取前 30% 作为"正样本"
labels = (true_scores > np.quantile(true_scores, 0.7)).astype(int)
scores_pred = true_scores + rng.normal(0, 0.3, (n_users, n_items))  # 预测分=真值+噪声
print('标签矩阵形状:', labels.shape, '正样本占比:', labels.mean().round(3))
assert 0.25 < labels.mean() < 0.35
print('✅ 数据就绪（含少量噪声的预测分）')

标签矩阵形状: (100, 50) 正样本占比: 0.3
✅ 数据就绪（含少量噪声的预测分）


## 3️⃣ AUC 手写 vs sklearn

In [4]:
from sklearn.metrics import roc_auc_score

y_true = labels.ravel()
y_score = scores_pred.ravel()

# 手写 AUC：随机正样本得分 > 随机负样本得分的概率（秩和法）
def auc_manual(y, s):
    order = np.argsort(s)
    rank = np.empty_like(order)
    rank[order] = np.arange(len(s)) + 1
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    rank_sum_pos = rank[y == 1].sum()
    return (rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

auc_m = auc_manual(y_true, y_score)
auc_s = roc_auc_score(y_true, y_score)
print(f'手写 AUC = {auc_m:.4f}')
print(f'sklearn AUC = {auc_s:.4f}')
assert abs(auc_m - auc_s) < 1e-9
print('✅ 手写与 sklearn 完全一致')

# 可视化 ROC 曲线
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_true, y_score)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='#2563eb', lw=2, label=f'AUC={auc_s:.3f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='随机')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC 曲线')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

手写 AUC = 0.9728
sklearn AUC = 0.9728
✅ 手写与 sklearn 完全一致


C:\Users\24260\AppData\Local\Temp\ipykernel_29424\4099093705.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4️⃣ GAUC：用户级 AUC（加权平均）

In [5]:
# GAUC = 按用户分别算 AUC，再按各用户样本数加权
gauc_vals = []
for u in range(n_users):
    y_u = labels[u]
    s_u = scores_pred[u]
    if y_u.sum() == 0 or y_u.sum() == len(y_u):
        continue  # 纯正/纯负用户无 AUC
    gauc_vals.append((y_u.sum(), auc_manual(y_u, s_u)))
w = np.array([n for n, _ in gauc_vals])
a = np.array([v for _, v in gauc_vals])
gauc = (a * w).sum() / w.sum()
print(f'GAUC = {gauc:.4f}（按用户样本量加权）')
print(f'说明：GAUC 更贴近推荐场景（每个用户一个排序列表），比全局 AUC 对"用户间差异"更敏感')
assert 0.5 < gauc < 1.0
print('✅ GAUC 计算完成')

GAUC = 0.9730（按用户样本量加权）
说明：GAUC 更贴近推荐场景（每个用户一个排序列表），比全局 AUC 对"用户间差异"更敏感
✅ GAUC 计算完成


## 5️⃣ HR@K / Recall@K / NDCG@K

In [6]:
def hr_recall_ndcg(rel_matrix, K):
    # rel_matrix: (users, items) 0/1 相关性；对每用户取预测分最高的 K 个
    hr_list, rec_list, ndcg_list = [], [], []
    for u in range(rel_matrix.shape[0]):
        rel = rel_matrix[u]
        n_rel = rel.sum()
        if n_rel == 0:
            continue
        top_k_idx = np.argsort(-scores_pred[u])[:K]
        hit = rel[top_k_idx].sum()
        hr_list.append(1.0 if hit > 0 else 0.0)
        rec_list.append(hit / n_rel)
        # NDCG@K
        dcg = sum((2**rel[top_k_idx[i]] - 1) / np.log2(i + 2) for i in range(K))
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(n_rel, K)))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(hr_list), np.mean(rec_list), np.mean(ndcg_list)

K = 10
hr, rec, ndcg = hr_recall_ndcg(labels, K)
print(f'HR@{K}  = {hr:.4f}（至少命中 1 个相关物品的用户比例）')
print(f'Recall@{K} = {rec:.4f}（相关物品被召回的比例均值）')
print(f'NDCG@{K} = {ndcg:.4f}（位置加权：排得越靠前越加分）')
assert 0 < hr <= 1 and 0 < ndcg <= 1
print('✅ 推荐列表三指标计算完成')

HR@10  = 1.0000（至少命中 1 个相关物品的用户比例）
Recall@10 = 0.6455（相关物品被召回的比例均值）
NDCG@10 = 0.9615（位置加权：排得越靠前越加分）
✅ 推荐列表三指标计算完成


## 6️⃣ 离线评估 vs 在线评估（必背）

In [7]:
print('''离线评估：在历史数据上回放（GAUC/NDCG/Recall@K），快但存在偏差：
  1) 曝光偏差：只有被曝光的物品才有反馈；
  2) 位置偏差：排在前面的天然有点击优势；
  3) 数据分布随策略变化：训练分布≠线上分布（反事实偏差）。

在线评估：小流量 AB 实验，看线上指标（CTR/CVR/停留/时长/GMV/留存），
  最可信但慢、贵、有风险（打错伤体验）。

工业做法：离线筛选 → 在线小流量 AB → 逐步放量 → 全量。
''')

离线评估：在历史数据上回放（GAUC/NDCG/Recall@K），快但存在偏差：
  1) 曝光偏差：只有被曝光的物品才有反馈；
  2) 位置偏差：排在前面的天然有点击优势；
  3) 数据分布随策略变化：训练分布≠线上分布（反事实偏差）。

在线评估：小流量 AB 实验，看线上指标（CTR/CVR/停留/时长/GMV/留存），
  最可信但慢、贵、有风险（打错伤体验）。

工业做法：离线筛选 → 在线小流量 AB → 逐步放量 → 全量。



## 📝 自测清单

- [ ] 能画出推荐五段链路并讲每层职责与时延预算
- [ ] 能手写 AUC（秩和法）并解释其物理意义
- [ ] 能解释 GAUC 与全局 AUC 的区别与适用场景
- [ ] 能手写 HR@K / Recall@K / NDCG@K
- [ ] 能说出离线评估三大偏差与在线 AB 的取舍